In [0]:
df = spark.table("workspace.default.bronze_sales")
display(df)

In [0]:
from pyspark.sql.functions import *

silver_df = df.filter(col("amount").isNotNull()) \
.filter(col("quantity").isNotNull()) \
.withColumn("amount", col("amount").cast("double")) \
.withColumn("product_id",upper(col("product_id"))) \
.withColumn("quantity", col("quantity").cast("int")) # DropDuplicates was removed (no redundant aggregation)

In [0]:
reject_df=silver_df.filter(col("amount")>=0)
display(reject_df)

In [0]:
reject_df.write.mode("overwrite").saveAsTable("workspace.default.reject_df") # No code change needed; aggregation fix was in preprocessing

In [0]:
silver_df.write.mode("overwrite").saveAsTable("workspace.default.silver_sales")

In [0]:
%sql
select * from workspace.default.silver_sales

In [0]:
from pyspark.sql.functions import *
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.default.pipeline_audit (
    run_id STRING,
    table_name STRING,
    source_count BIGINT,
    target_count BIGINT,
    reject_count BIGINT,
    load_timestamp TIMESTAMP
)
USING DELTA
""")


In [0]:
%sql
select * from workspace.default.pipeline_audit


In [0]:
source_count=spark.sql("select count(*) from workspace.default.silver_sales")
reject_count=spark.sql("select count(*) from workspace.default.reject_df")

In [0]:
from uuid import uuid4

run_id = str(uuid4())


In [0]:
%sql
select * from workspace.default.reject_df

In [0]:
%sql
select * from workspace.default.pipeline_audit

In [0]:
spark.sql(f"""
INSERT INTO workspace.default.pipeline_audit
(
    run_id,
    table_name,
    source_count,
    target_count,
    reject_count,
    load_timestamp
)
VALUES
(
    '{run_id}',
    'customers',
    {source_count.collect()[0][0]},
    0,
    {reject_count.collect()[0][0]},
    current_timestamp()
)
""")

In [0]:
%sql
select * from workspace.default.pipeline_audit